# Step 2 - Training the three models

Multi-label toxicity classification (6 labels) on the cleaned, anonymised comments
(`processed_data/train_clean.csv`). Three families are trained and compared on the same split:

| Model | Family | Notes |
|---|---|---|
| `lstm` | statistical / neural | Bidirectional LSTM trained from scratch (PyTorch, CUDA) |
| `bert` | transformer | Fine-tuned `bert-base-uncased` (Hugging Face, CUDA, mixed precision) |
| `xgboost` | decision model | Gradient-boosted trees on TF-IDF features (XGBoost, CUDA) |

Every model reports precision, recall, F1 and ROC-AUC per label, plus training and inference time.
Decision thresholds are tuned on the validation split (labels are very imbalanced), never on test.

**CUDA is required**: the notebook stops if no GPU is visible.

## 0. Setup

In [2]:
import json
import logging
import math
import random
import time
from collections import Counter
from collections.abc import Iterator, Sequence
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
import torch
from numpy.typing import NDArray
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch import Tensor, nn
from torch.amp.grad_scaler import GradScaler
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from xgboost import XGBClassifier

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True
)
logger = logging.getLogger("train")

# Set to a small number (e.g. 5000) for a quick smoke test, None for the full run.
LIMIT: int | None = None

SEED = 42
VALIDATION_SIZE = 0.1
TEXT_COLUMN = "comment_text"
LABELS: tuple[str, ...] = (
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate",
)

FloatArray = NDArray[np.float32]
IntArray = NDArray[np.int64]


def find_project_root() -> Path:
    """Walk up from the working directory until pyproject.toml is found."""
    for parent in (Path.cwd(), *Path.cwd().parents):
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("pyproject.toml not found above the working directory")


PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "raw_data"
PROCESSED_DATA_DIR = PROJECT_ROOT / "processed_data"
MODELS_DIR = PROJECT_ROOT / "models"

## 1. Helpers: CUDA check, data loading, thresholds and metrics

Thresholds maximise F1 per label on the validation split. `finalize` evaluates on the test split
(rows Kaggle scores, i.e. labels different from `-1`) and writes `models/<name>/metrics.json`.

In [3]:
def set_seed(seed: int = SEED) -> None:
    """Seed every random number generator used in this notebook."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def require_cuda() -> torch.device:
    """Return the CUDA device, or fail loudly: training on CPU is not supported here."""
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. Check the driver with `nvidia-smi` and make sure the "
            "CUDA build of PyTorch is installed (`uv sync`, index pytorch-cu128)."
        )
    logger.info("Using CUDA device: %s", torch.cuda.get_device_name(0))
    return torch.device("cuda")


def read_clean(path: Path) -> pd.DataFrame:
    """Read a cleaned CSV; comments that became empty are read back as NaN."""
    df = pd.read_csv(path)
    df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str)
    return df


def load_train_val(limit: int | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load the cleaned training set and split it into train and validation parts."""
    df = read_clean(PROCESSED_DATA_DIR / "train_clean.csv")
    if limit is not None:
        df = df.head(limit)
    train_df, val_df = train_test_split(
        df, test_size=VALIDATION_SIZE, random_state=SEED, stratify=df["toxic"]
    )
    logger.info("Train rows: %d, validation rows: %d", len(train_df), len(val_df))
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


def load_test(limit: int | None = None) -> pd.DataFrame:
    """Load the cleaned test set joined with its labels (rows labelled -1 are not scored)."""
    df = read_clean(PROCESSED_DATA_DIR / "test_clean.csv")
    df = df.drop(columns=list(LABELS), errors="ignore")
    labels = pd.read_csv(RAW_DATA_DIR / "test_labels.csv")
    merged = df.merge(labels, on="id", how="inner")
    merged = merged[merged["toxic"] != -1].reset_index(drop=True)
    if limit is not None:
        merged = merged.head(limit)
    logger.info("Scored test rows: %d", len(merged))
    return merged


def labels_array(df: pd.DataFrame) -> IntArray:
    """Return the label columns as an (n_rows, 6) integer array."""
    return df[list(LABELS)].to_numpy(dtype=np.int64)


def mean_auc(y_true: IntArray, y_prob: FloatArray) -> float:
    """Mean ROC-AUC over the labels (early stopping criterion)."""
    return float(np.mean([roc_auc_score(y_true[:, i], y_prob[:, i]) for i in range(len(LABELS))]))


def find_thresholds(y_true: IntArray, y_prob: FloatArray) -> list[float]:
    """Pick, per label, the threshold that maximises F1 (default 0.5 gives poor recall)."""
    thresholds: list[float] = []
    for i in range(len(LABELS)):
        precision, recall, candidates = precision_recall_curve(y_true[:, i], y_prob[:, i])
        f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
        thresholds.append(float(candidates[int(np.argmax(f1))]) if len(candidates) else 0.5)
    return thresholds


def evaluate(y_true: IntArray, y_prob: FloatArray, thresholds: list[float]) -> dict[str, Any]:
    """Precision, recall, F1 and ROC-AUC per label, plus their macro average."""
    per_label: dict[str, dict[str, float]] = {}
    for i, label in enumerate(LABELS):
        predicted = (y_prob[:, i] >= thresholds[i]).astype(np.int64)
        precision = float(precision_score(y_true[:, i], predicted, zero_division=0))
        recall = float(recall_score(y_true[:, i], predicted, zero_division=0))
        f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
        per_label[label] = {
            "threshold": thresholds[i],
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "roc_auc": float(roc_auc_score(y_true[:, i], y_prob[:, i])),
        }
    macro = {
        metric: float(np.mean([scores[metric] for scores in per_label.values()]))
        for metric in ("precision", "recall", "f1", "roc_auc")
    }
    return {"per_label": per_label, "macro": macro}


def finalize(
    name: str,
    y_val: IntArray,
    val_prob: FloatArray,
    y_test: IntArray,
    test_prob: FloatArray,
    train_seconds: float,
    inference_seconds: float,
    config: dict[str, Any],
) -> dict[str, Any]:
    """Tune thresholds on validation, evaluate on test and write metrics.json."""
    thresholds = find_thresholds(y_val, val_prob)
    results: dict[str, Any] = evaluate(y_test, test_prob, thresholds)
    results["model"] = name
    results["train_seconds"] = round(train_seconds, 1)
    results["inference_seconds"] = round(inference_seconds, 2)
    results["inference_ms_per_comment"] = round(1000 * inference_seconds / len(y_test), 4)
    results["test_rows"] = len(y_test)
    results["config"] = config
    out_dir = MODELS_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "metrics.json").write_text(
        json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    macro = results["macro"]
    logger.info(
        "[%s] macro precision=%.4f recall=%.4f f1=%.4f roc_auc=%.4f | train %.0fs | "
        "inference %.3f ms/comment",
        name,
        macro["precision"],
        macro["recall"],
        macro["f1"],
        macro["roc_auc"],
        train_seconds,
        results["inference_ms_per_comment"],
    )
    return results

## 2. Load the data and check CUDA

In [4]:
set_seed()
device = require_cuda()

train_df, val_df = load_train_val(LIMIT)
test_df = load_test(LIMIT)
y_train = labels_array(train_df)
y_val = labels_array(val_df)
y_test = labels_array(test_df)

pd.DataFrame(
    {
        "train": y_train.mean(axis=0),
        "validation": y_val.mean(axis=0),
        "test": y_test.mean(axis=0),
    },
    index=list(LABELS),
).round(4)

2026-11-10 11:25:16,376 INFO train: Using CUDA device: NVIDIA GeForce RTX 3050 6GB Laptop GPU
2026-11-10 11:25:17,113 INFO train: Train rows: 143613, validation rows: 15958
2026-11-10 11:25:17,478 INFO train: Scored test rows: 63978


,train,validation,test
toxic,0.0958,0.0958,0.0952
severe_toxic,0.0101,0.0093,0.0057
obscene,0.0529,0.0531,0.0577
threat,0.0030,0.0031,0.0033
insult,0.0493,0.0501,0.0536
identity_hate,0.0087,0.0096,0.0111


## 3. Model 1 - BiLSTM (statistical / neural)

Tokens are the whitespace-separated words of the cleaned text (already lower-cased, no punctuation).
The vocabulary keeps the 50,000 most frequent words seen at least 3 times; the others map to `<unk>`.
The network is `Embedding -> BiLSTM -> [max pooling + mean pooling] -> Linear(6)`, trained with a
binary cross-entropy loss per label and mixed precision on the GPU.

In [5]:
LSTM_CONFIG: dict[str, Any] = {
    "max_vocab": 50_000,
    "min_freq": 3,
    "max_len": 200,
    "embed_dim": 128,
    "hidden_dim": 128,
    "dropout": 0.3,
    "epochs": 5,
    "batch_size": 256,
    "lr": 2e-3,
}
PAD_ID, UNK_ID = 0, 1


def build_vocab(texts: Sequence[str], max_vocab: int, min_freq: int) -> dict[str, int]:
    """Map the most frequent words to integer ids (0 = padding, 1 = unknown)."""
    counts = Counter(token for text in texts for token in text.split())
    vocab = {"<pad>": PAD_ID, "<unk>": UNK_ID}
    for token, count in counts.most_common(max_vocab - len(vocab)):
        if count < min_freq:
            break
        vocab[token] = len(vocab)
    return vocab


def encode_texts(
    texts: Sequence[str], vocab: dict[str, int], max_len: int
) -> tuple[Tensor, Tensor]:
    """Return padded token ids and true lengths (an empty comment counts as one unknown token)."""
    ids = torch.zeros((len(texts), max_len), dtype=torch.long)
    lengths = torch.ones(len(texts), dtype=torch.long)
    for row, text in enumerate(texts):
        tokens = [vocab.get(token, UNK_ID) for token in text.split()[:max_len]] or [UNK_ID]
        ids[row, : len(tokens)] = torch.tensor(tokens, dtype=torch.long)
        lengths[row] = len(tokens)
    return ids, lengths


class BiLstmClassifier(nn.Module):
    """Embedding -> BiLSTM -> max + mean pooling over real tokens -> 6 logits."""

    def __init__(
        self, vocab_size: int, embed_dim: int, hidden_dim: int, num_labels: int, dropout: float
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 4, num_labels)

    def forward(self, ids: Tensor, lengths: Tensor) -> Tensor:
        embedded = self.dropout(self.embedding(ids))
        packed = pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        output, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(output, batch_first=True, total_length=ids.size(1))
        positions = torch.arange(ids.size(1), device=ids.device).unsqueeze(0)
        mask = (positions < lengths.unsqueeze(1)).unsqueeze(-1)
        max_pool = output.masked_fill(~mask, float("-inf")).max(dim=1).values
        mean_pool = (output * mask).sum(dim=1) / lengths.unsqueeze(1)
        logits: Tensor = self.classifier(self.dropout(torch.cat([max_pool, mean_pool], dim=1)))
        return logits


def predict_lstm(
    model: BiLstmClassifier, ids: Tensor, lengths: Tensor, batch_size: int
) -> FloatArray:
    """Probabilities for every label, shape (n_rows, 6)."""
    model.eval()
    outputs: list[FloatArray] = []
    with torch.no_grad():
        for start in range(0, len(ids), batch_size):
            batch_ids = ids[start : start + batch_size].to(device)
            batch_lengths = lengths[start : start + batch_size].to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(batch_ids, batch_lengths)
            outputs.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(outputs).astype(np.float32)


def fit_lstm(
    model: BiLstmClassifier,
    train_data: tuple[Tensor, Tensor, IntArray],
    val_data: tuple[Tensor, Tensor, IntArray],
    config: dict[str, Any],
) -> BiLstmClassifier:
    """Train with AdamW + mixed precision; keep the epoch with the best validation ROC-AUC."""
    train_ids, train_lengths, train_labels = train_data
    val_ids, val_lengths, val_labels = val_data
    dataset = torch.utils.data.TensorDataset(
        train_ids, train_lengths, torch.from_numpy(train_labels).float()
    )
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=config["batch_size"], shuffle=True, pin_memory=True
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"])
    scaler = GradScaler("cuda")
    loss_fn = nn.BCEWithLogitsLoss()
    best_auc = -1.0
    best_state = {n: t.detach().cpu().clone() for n, t in model.state_dict().items()}
    for epoch in range(config["epochs"]):
        model.train()
        total_loss = 0.0
        for batch_ids, batch_lengths, batch_labels in loader:
            batch_ids = batch_ids.to(device, non_blocking=True)
            batch_lengths = batch_lengths.to(device, non_blocking=True)
            batch_labels = batch_labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(batch_ids, batch_lengths)
            loss = loss_fn(logits.float(), batch_labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.item())
        val_auc = mean_auc(val_labels, predict_lstm(model, val_ids, val_lengths, 1024))
        logger.info(
            "LSTM epoch %d/%d: train loss %.4f, validation ROC-AUC %.4f",
            epoch + 1,
            config["epochs"],
            total_loss / len(loader),
            val_auc,
        )
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {n: t.detach().cpu().clone() for n, t in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

In [6]:
set_seed()
start = time.perf_counter()
max_len = LSTM_CONFIG["max_len"]
vocab = build_vocab(
    train_df[TEXT_COLUMN].tolist(), LSTM_CONFIG["max_vocab"], LSTM_CONFIG["min_freq"]
)
logger.info("LSTM vocabulary size: %d", len(vocab))
train_ids, train_lengths = encode_texts(train_df[TEXT_COLUMN].tolist(), vocab, max_len)
val_ids, val_lengths = encode_texts(val_df[TEXT_COLUMN].tolist(), vocab, max_len)

lstm_model = BiLstmClassifier(
    len(vocab),
    LSTM_CONFIG["embed_dim"],
    LSTM_CONFIG["hidden_dim"],
    len(LABELS),
    LSTM_CONFIG["dropout"],
).to(device)
lstm_model = fit_lstm(
    lstm_model, (train_ids, train_lengths, y_train), (val_ids, val_lengths, y_val), LSTM_CONFIG
)
lstm_train_seconds = time.perf_counter() - start

lstm_val_prob = predict_lstm(lstm_model, val_ids, val_lengths, 1024)
start = time.perf_counter()  # inference time includes encoding the raw text
test_ids, test_lengths = encode_texts(test_df[TEXT_COLUMN].tolist(), vocab, max_len)
lstm_test_prob = predict_lstm(lstm_model, test_ids, test_lengths, 1024)
torch.cuda.synchronize()
lstm_inference_seconds = time.perf_counter() - start

lstm_results = finalize(
    "lstm", y_val, lstm_val_prob, y_test, lstm_test_prob,
    lstm_train_seconds, lstm_inference_seconds, LSTM_CONFIG,
)
lstm_dir = MODELS_DIR / "lstm"
torch.save(lstm_model.state_dict(), lstm_dir / "model.pt")
(lstm_dir / "vocab.json").write_text(json.dumps(vocab), encoding="utf-8")

del lstm_model
torch.cuda.empty_cache()

2026-11-10 11:33:39,838 INFO train: LSTM vocabulary size: 49978
C:\Users\houeg\AppData\Local\Temp\ipykernel_21224\611522123.py:92: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  train_ids, train_lengths, torch.from_numpy(train_labels).float()
2026-11-10 11:35:24,247 INFO train: LSTM epoch 1/5: train loss 0.0875, validation ROC-AUC 0.9655
2026-11-10 11:36:43,338 INFO train: LSTM epoch 2/5: train loss 0.0544, validation ROC-AUC 0.9755
2026-11-10 11:38:00,423 INFO train: LSTM epoch 3/5: train loss 0.0472, validation ROC-AUC 0.9802
2026-11-10 11:39:31,621 INFO train: LSTM epoch 4/5:

## 4. Model 2 - BERT (transformer)

`bert-base-uncased` fine-tuned for multi-label classification (the cleaned text is lower-cased, so the
uncased model fits). Sequences are truncated to 128 tokens and padded per batch. The batch size of 16
with mixed precision is chosen for a 6 GB GPU (RTX 3050); lower `batch_size` or `max_len` if memory runs
out, or set `model_name` to `distilbert-base-uncased` for a lighter and faster run.

In [ ]:
BERT_CONFIG: dict[str, Any] = {
    "model_name": "bert-base-uncased",
    "max_len": 128,
    "epochs": 2,
    "batch_size": 16,
    "eval_batch_size": 64,
    "lr": 2e-5,
    "warmup_ratio": 0.06,
}


def iter_batches(
    input_ids: list[list[int]],
    labels: IntArray | None,
    batch_size: int,
    pad_id: int,
    rng: np.random.Generator | None,
) -> Iterator[tuple[Tensor, Tensor, Tensor | None]]:
    """Yield padded (input_ids, attention_mask, labels) batches; shuffled when rng is given."""
    order = rng.permutation(len(input_ids)) if rng is not None else np.arange(len(input_ids))
    for start in range(0, len(order), batch_size):
        indices = order[start : start + batch_size]
        longest = max(len(input_ids[i]) for i in indices)
        ids = torch.full((len(indices), longest), pad_id, dtype=torch.long)
        mask = torch.zeros((len(indices), longest), dtype=torch.long)
        for row, i in enumerate(indices):
            ids[row, : len(input_ids[i])] = torch.tensor(input_ids[i], dtype=torch.long)
            mask[row, : len(input_ids[i])] = 1
        batch_labels = None if labels is None else torch.from_numpy(labels[indices]).float()
        yield ids, mask, batch_labels


def predict_bert(
    model: Any, input_ids: list[list[int]], pad_id: int, batch_size: int
) -> FloatArray:
    """Probabilities for every label, shape (n_rows, 6), in the input order."""
    model.eval()
    outputs: list[FloatArray] = []
    with torch.no_grad():
        for ids, mask, _ in iter_batches(input_ids, None, batch_size, pad_id, None):
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            outputs.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(outputs).astype(np.float32)


def fit_bert(
    model: Any,
    train_ids: list[list[int]],
    train_labels: IntArray,
    val_ids: list[list[int]],
    val_labels: IntArray,
    pad_id: int,
    config: dict[str, Any],
    save_dir: Path,
) -> None:
    """Fine-tune with AdamW, linear warm-up/decay and mixed precision; save the best epoch."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=0.01)
    steps_per_epoch = math.ceil(len(train_ids) / config["batch_size"])
    total_steps = steps_per_epoch * config["epochs"]
    scheduler = get_linear_schedule_with_warmup(  # type: ignore[no-untyped-call]
        optimizer, int(config["warmup_ratio"] * total_steps), total_steps
    )
    scaler = GradScaler("cuda")
    loss_fn = nn.BCEWithLogitsLoss()
    best_auc = -1.0
    for epoch in range(config["epochs"]):
        model.train()
        rng = np.random.default_rng(SEED + epoch)
        running_loss = 0.0
        batches = iter_batches(train_ids, train_labels, config["batch_size"], pad_id, rng)
        for step, (ids, mask, batch_labels) in enumerate(batches, start=1):
            assert batch_labels is not None
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            loss = loss_fn(logits.float(), batch_labels.to(device))
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += float(loss.item())
            if step % 500 == 0:
                logger.info(
                    "BERT epoch %d step %d/%d: loss %.4f",
                    epoch + 1, step, steps_per_epoch, running_loss / step,
                )
        val_prob = predict_bert(model, val_ids, pad_id, config["eval_batch_size"])
        val_auc = mean_auc(val_labels, val_prob)
        logger.info(
            "BERT epoch %d/%d: validation ROC-AUC %.4f", epoch + 1, config["epochs"], val_auc
        )
        if val_auc > best_auc:
            best_auc = val_auc
            model.save_pretrained(save_dir)

In [ ]:
set_seed()
bert_dir = MODELS_DIR / "bert"
bert_dir.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(BERT_CONFIG["model_name"])
pad_id = int(tokenizer.pad_token_id)


def tokenize(texts: list[str]) -> list[list[int]]:
    """Token ids truncated to max_len (special tokens included, no padding yet)."""
    encoded = tokenizer(texts, truncation=True, max_length=BERT_CONFIG["max_len"])
    return [list(ids) for ids in encoded["input_ids"]]


start = time.perf_counter()
bert_train_ids = tokenize(train_df[TEXT_COLUMN].tolist())
bert_val_ids = tokenize(val_df[TEXT_COLUMN].tolist())

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_CONFIG["model_name"], num_labels=len(LABELS), problem_type="multi_label_classification"
).to(device)
fit_bert(bert_model, bert_train_ids, y_train, bert_val_ids, y_val, pad_id, BERT_CONFIG, bert_dir)
bert_train_seconds = time.perf_counter() - start

# Reload the best epoch saved on disk.
bert_model = AutoModelForSequenceClassification.from_pretrained(bert_dir).to(device)
tokenizer.save_pretrained(bert_dir)
bert_val_prob = predict_bert(bert_model, bert_val_ids, pad_id, BERT_CONFIG["eval_batch_size"])

start = time.perf_counter()  # inference time includes tokenisation of the raw text
bert_test_ids = tokenize(test_df[TEXT_COLUMN].tolist())
bert_test_prob = predict_bert(bert_model, bert_test_ids, pad_id, BERT_CONFIG["eval_batch_size"])
torch.cuda.synchronize()
bert_inference_seconds = time.perf_counter() - start

bert_results = finalize(
    "bert", y_val, bert_val_prob, y_test, bert_test_prob,
    bert_train_seconds, bert_inference_seconds, BERT_CONFIG,
)

del bert_model
torch.cuda.empty_cache()

## 5. Model 3 - Gradient-boosted decision trees (decision model)

Text is turned into TF-IDF features (unigrams and bigrams, 20,000 features to fit in 6 GB of GPU memory),
then one XGBoost classifier is trained per label on the GPU with early stopping on the validation split.
`scale_pos_weight` (square root of the negative/positive ratio) compensates the class imbalance.

In [ ]:
TREE_CONFIG: dict[str, Any] = {
    "max_features": 20_000,
    "ngram_range": (1, 2),
    "min_df": 3,
    "n_estimators": 1000,
    "learning_rate": 0.1,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.5,
    "early_stopping_rounds": 30,
}

set_seed()
start = time.perf_counter()
vectorizer = TfidfVectorizer(
    max_features=TREE_CONFIG["max_features"],
    ngram_range=TREE_CONFIG["ngram_range"],
    min_df=TREE_CONFIG["min_df"],
    sublinear_tf=True,
    dtype=np.float32,
)
x_train = vectorizer.fit_transform(train_df[TEXT_COLUMN])
x_val = vectorizer.transform(val_df[TEXT_COLUMN])

boosters: list[XGBClassifier] = []
for index, label in enumerate(LABELS):
    positives = int(y_train[:, index].sum())
    negatives = len(y_train) - positives
    booster = XGBClassifier(
        n_estimators=TREE_CONFIG["n_estimators"],
        learning_rate=TREE_CONFIG["learning_rate"],
        max_depth=TREE_CONFIG["max_depth"],
        subsample=TREE_CONFIG["subsample"],
        colsample_bytree=TREE_CONFIG["colsample_bytree"],
        scale_pos_weight=float(np.sqrt(negatives / max(positives, 1))),
        tree_method="hist",
        device="cuda",
        eval_metric="auc",
        early_stopping_rounds=TREE_CONFIG["early_stopping_rounds"],
        random_state=SEED,
    )
    booster.fit(x_train, y_train[:, index], eval_set=[(x_val, y_val[:, index])], verbose=False)
    logger.info("XGBoost '%s': best iteration %s", label, booster.best_iteration)
    boosters.append(booster)
tree_train_seconds = time.perf_counter() - start


def predict_tree(texts: list[str]) -> FloatArray:
    """Probabilities for every label, shape (n_rows, 6)."""
    features = vectorizer.transform(texts)
    columns = [booster.predict_proba(features)[:, 1] for booster in boosters]
    return np.column_stack(columns).astype(np.float32)


tree_val_prob = predict_tree(val_df[TEXT_COLUMN].tolist())
start = time.perf_counter()  # inference time includes vectorising the raw text
tree_test_prob = predict_tree(test_df[TEXT_COLUMN].tolist())
tree_inference_seconds = time.perf_counter() - start

tree_results = finalize(
    "xgboost", y_val, tree_val_prob, y_test, tree_test_prob,
    tree_train_seconds, tree_inference_seconds, TREE_CONFIG,
)
tree_dir = MODELS_DIR / "xgboost"
joblib.dump(vectorizer, tree_dir / "tfidf.joblib")
for label, booster in zip(LABELS, boosters, strict=True):
    booster.save_model(tree_dir / f"{label}.json")

## 6. Comparison

Macro-averaged metrics on the scored test rows, plus training time and inference cost.

In [ ]:
rows: list[dict[str, Any]] = []
for model_name in ("lstm", "bert", "xgboost"):
    metrics_path = MODELS_DIR / model_name / "metrics.json"
    if not metrics_path.exists():
        continue
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    rows.append(
        {
            "model": model_name,
            **metrics["macro"],
            "train_seconds": metrics["train_seconds"],
            "inference_ms_per_comment": metrics["inference_ms_per_comment"],
        }
    )
pd.DataFrame(rows).set_index("model").round(4)

In [ ]:
per_label: dict[str, pd.DataFrame] = {}
for model_name in ("lstm", "bert", "xgboost"):
    metrics_path = MODELS_DIR / model_name / "metrics.json"
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        per_label[model_name] = pd.DataFrame(metrics["per_label"]).T[
            ["precision", "recall", "f1", "roc_auc"]
        ]
pd.concat(per_label, names=["model", "label"]).round(4)